# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveenadanthapally/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I will use a **Random Forest Regressor** to estimate the next observed CTR from information available at the current observation.

This method fits the lane because the question is about identifying pages with potential future search-performance improvement. The warehouse does not contain a direct refresh-success label, so I will use the next observed CTR as a limited future-performance proxy rather than calling it a true refresh outcome.

Random Forest is appropriate because it can model nonlinear relationships between search visibility and engagement signals without requiring a linear relationship. I will keep the model simple and compare it directly with the transparent Week-4 baseline.

The model is used for **decision-support**, not causal inference. A higher predicted future CTR does not prove that refreshing a page will cause an improvement.


In [30]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance

print("Model: Random Forest Regressor")
print("Target: next observed CTR")
print("Purpose: directional decision-support")

Model: Random Forest Regressor
Target: next observed CTR
Purpose: directional decision-support


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I will use a **time-aware split** because the same content pages can appear on multiple report dates.

The target for each observation is the CTR on the next observed report date for the same client and content page. Therefore, the model must only use information available at the current report date.

I will use the earlier observed dates for training and the later observed dates for testing. This avoids using future observations to predict earlier observations.

The date gap from January 31 to February 10 is retained rather than treated as continuous daily history. The outcome means the next available observed date, not necessarily the following calendar day.


In [31]:
# Make sure dates and numeric fields are correctly typed.
df["report_date"] = pd.to_datetime(df["report_date"])

numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events",
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

print("Observed dates:", sorted(df["report_date"].dt.strftime("%Y-%m-%d").unique()))

Observed dates: ['2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-10', '2025-02-11', '2025-02-12', '2025-02-13', '2025-02-14']


In [32]:
# Calculate observed CTR.
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0.0
)

# For each page/client pair, obtain the CTR from the next observed date.
group_cols = ["client_hash_id", "content_hash_id"]

df["future_ctr"] = (
    df.groupby(group_cols)["ctr"]
      .shift(-1)
)

df["future_date"] = (
    df.groupby(group_cols)["report_date"]
      .shift(-1)
)

model_df = df.dropna(subset=["future_ctr"]).copy()

print("Rows with a future observed CTR:", len(model_df))
print("Rows removed because no future observation exists:",
      len(df) - len(model_df))

Rows with a future observed CTR: 6479
Rows removed because no future observation exists: 3521


In [33]:
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other",
    "scroll_events",
]

feature_cols = [
    c for c in feature_cols
    if c in model_df.columns
]

print("Number of model features:", len(feature_cols))
print(feature_cols)

# Keep only rows where the required features are available.
model_df = model_df.dropna(
    subset=feature_cols + ["future_ctr"]
).copy()

print("Rows available for modeling:", len(model_df))

Number of model features: 22
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Rows available for modeling: 6479


In [34]:
unique_dates = sorted(model_df["report_date"].unique())

split_index = max(1, int(len(unique_dates) * 0.7))

train_dates = unique_dates[:split_index]
test_dates = unique_dates[split_index:]

train_df = model_df[
    model_df["report_date"].isin(train_dates)
].copy()

test_df = model_df[
    model_df["report_date"].isin(test_dates)
].copy()

print("Training dates:")
print([d.strftime("%Y-%m-%d") for d in train_dates])

print("\nTest dates:")
print([d.strftime("%Y-%m-%d") for d in test_dates])

print("\nTraining rows:", len(train_df))
print("Test rows:", len(test_df))

Training dates:
['2025-01-27', '2025-01-28', '2025-01-29', '2025-01-30', '2025-01-31', '2025-02-10']

Test dates:
['2025-02-11', '2025-02-12', '2025-02-13']

Training rows: 2145
Test rows: 4334


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest will be trained only on observations from the earlier time period. The test set contains later report dates that were not used during training.

The comparison uses the same held-out observations. The model produces a predicted future CTR, while the Week-4 baseline produces its existing action score.

Because the baseline is a ranking score rather than a calibrated prediction, I will compare both methods primarily as ranking systems on the same test observations. I will also report the model's MAE and RMSE for the future-CTR prediction task.


In [35]:
X_train = train_df[feature_cols]
y_train = train_df["future_ctr"]

X_test = test_df[feature_cols]
y_test = test_df["future_ctr"]

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

test_df["model_predicted_future_ctr"] = model.predict(X_test)

mae = mean_absolute_error(
    y_test,
    test_df["model_predicted_future_ctr"]
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        test_df["model_predicted_future_ctr"]
    )
)

print("Model MAE:", round(mae, 4))
print("Model RMSE:", round(rmse, 4))

Model MAE: 0.0202
Model RMSE: 0.0527


In [36]:
baseline_df = test_df.copy()

baseline_df["position_band"] = pd.cut(
    baseline_df["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)

baseline_df["ctr"] = np.where(
    baseline_df["gsc_impressions"] > 0,
    baseline_df["gsc_clicks"] / baseline_df["gsc_impressions"],
    0.0
)

band_ctr_median = (
    baseline_df
    .groupby("position_band", observed=True)["ctr"]
    .median()
)

baseline_df["band_ctr_median"] = (
    baseline_df["position_band"]
    .map(band_ctr_median)
    .astype(float)
)

baseline_df["ctr_opportunity"] = np.maximum(
    baseline_df["band_ctr_median"] - baseline_df["ctr"],
    0
)

# Normalize impression visibility.
imp_min = baseline_df["gsc_impressions"].min()
imp_max = baseline_df["gsc_impressions"].max()

if imp_max > imp_min:
    baseline_df["impression_visibility"] = (
        baseline_df["gsc_impressions"] - imp_min
    ) / (imp_max - imp_min)
else:
    baseline_df["impression_visibility"] = 0.0

# Week-4 baseline score.
baseline_df["baseline_action_score"] = (
    0.70 * baseline_df["ctr_opportunity"]
    + 0.30 * baseline_df["impression_visibility"]
)

display(
    baseline_df[
        [
            "report_date",
            "content_hash_id",
            "baseline_action_score",
            "model_predicted_future_ctr",
            "future_ctr"
        ]
    ].head(10)
)

,report_date,content_hash_id,baseline_action_score,model_predicted_future_ctr,future_ctr
0,2025-02-12,content_00033c286cc93446,0.001188,0.002870,0.000000
3,2025-02-12,content_006bf4ecc0d180be,0.006535,0.006601,0.000000
6,2025-02-12,content_01069c2e5ac3ab8f,0.000594,0.005286,0.000000
8,2025-02-12,content_0124f3b2a9349f0b,0.000594,0.004894,0.000000
10,2025-02-12,content_012d2b91814389fd,0.007723,0.006467,0.000000
12,2025-02-12,content_0156b9520d225af7,0.000594,0.000703,0.000000
14,2025-02-11,content_017f96fd8aa8f84b,0.001782,0.013358,0.063636
15,2025-02-12,content_017f96fd8aa8f84b,0.064752,0.018362,0.010204
17,2025-02-12,content_018e6da4439c0048,0.004158,0.008843,0.000000
21,2025-02-11,content_021d85c6dd85504c,0.000594,0.013001,0.000000


In [37]:
from sklearn.metrics import ndcg_score

comparison_df = baseline_df.dropna(
    subset=[
        "baseline_action_score",
        "model_predicted_future_ctr",
        "future_ctr"
    ]
).copy()

y_true = comparison_df["future_ctr"].to_numpy().reshape(1, -1)

baseline_scores = comparison_df[
    "baseline_action_score"
].to_numpy().reshape(1, -1)

model_scores = comparison_df[
    "model_predicted_future_ctr"
].to_numpy().reshape(1, -1)

baseline_ndcg = ndcg_score(
    y_true,
    baseline_scores
)

model_ndcg = ndcg_score(
    y_true,
    model_scores
)

comparison_table = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Week-5 Random Forest"
    ],
    "NDCG": [
        baseline_ndcg,
        model_ndcg
    ]
})

comparison_table["NDCG"] = comparison_table["NDCG"].round(4)

display(comparison_table)

,Method,NDCG
0,Week-4 baseline,0.4391
1,Week-5 Random Forest,0.4629


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model's errors are examined using absolute prediction error rather than looking only at the overall metric.

Large errors indicate observations where the current measured signals did not accurately estimate the next observed CTR. These cases may reflect noisy low-impression observations, changes between observed dates, or other factors not represented in the available fields.

I will also use permutation importance to understand which observed features contribute most to the model's predictions. These are associations used for interpretation, not causal effects.


In [38]:
comparison_df["absolute_error"] = (
    comparison_df["future_ctr"]
    - comparison_df["model_predicted_future_ctr"]
).abs()

error_cols = [
    "report_date",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "future_ctr",
    "model_predicted_future_ctr",
    "absolute_error"
]

display(
    comparison_df
    .sort_values("absolute_error", ascending=False)
    [error_cols]
    .head(10)
)

,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,future_ctr,model_predicted_future_ctr,absolute_error
24,2025-02-11,content_0264e9a6433092d0,3,0,40.666667,1.0,0.002870,0.997130
2307,2025-02-12,content_d240342908ef8a43,6,0,6.333333,1.0,0.008083,0.991917
4588,2025-02-11,content_44f220c212f8f2a6,5,0,12.200000,1.0,0.012045,0.987955
6593,2025-02-12,content_942210a707f08ab3,2,0,17.500000,1.0,0.015212,0.984788
8527,2025-02-12,content_df4d07f2dacab000,1,0,27.000000,0.5,0.005851,0.494149
4578,2025-02-11,content_440621aab72727ea,2,0,6.000000,0.5,0.008678,0.491322
6084,2025-02-13,content_7ec8a10e4dbfe74e,4,0,6.000000,0.5,0.009173,0.490827
1146,2025-02-12,content_64ef3823cb4711cf,1,0,8.000000,0.5,0.009958,0.490042
155,2025-02-12,content_0e69a58e1bed66b6,3,0,4.333333,0.5,0.018413,0.481587
2218,2025-02-12,content_cb67ca92aca85cc0,3,1,0.666667,0.5,0.035968,0.464032


In [39]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

display(importance_df.head(10))

,feature,importance_mean,importance_std
2,gsc_avg_position,3.376645e-04,1.521264e-04
1,gsc_clicks,1.493979e-04,4.797038e-05
10,sessions_referral,-1.387779e-18,1.699675e-18
8,sessions_organic,-2.081668e-18,1.699675e-18
20,ai_other,-2.081668e-18,1.699675e-18
3,ga4_pageviews,-2.428613e-18,1.589900e-18
18,ai_claude,-2.428613e-18,1.589900e-18
13,sessions_ai,-2.428613e-18,1.589900e-18
17,ai_copilot,-2.428613e-18,1.589900e-18
7,ga4_total_engagement_sec,-2.428613e-18,1.589900e-18


In [40]:
top_features = importance_df.head(5)["feature"].tolist()

print("Top observed features by permutation importance:")
for i, feature in enumerate(top_features, start=1):
    print(f"{i}. {feature}")

print("\nLargest model errors:")
print(
    comparison_df["absolute_error"]
    .describe()
    .round(4)
)

Top observed features by permutation importance:
1. gsc_avg_position
2. gsc_clicks
3. sessions_referral
4. sessions_organic
5. ai_other

Largest model errors:
count    4318.0000
mean        0.0201
std         0.0487
min         0.0000
25%         0.0033
50%         0.0086
75%         0.0163
max         0.9971
Name: absolute_error, dtype: float64


### Interpretation

The Week-5 Random Forest achieved an NDCG of **0.4629** on the held-out test period, compared with **0.4391** for the Week-4 baseline. This is an improvement of **0.0238 NDCG points**.

This indicates that, on this test set, the Random Forest produced a somewhat better ranking of observations by their next observed CTR than the transparent Week-4 baseline.

The improvement should be interpreted cautiously. The target is the **next observed CTR**, which is only a limited proxy for future search performance and is not a direct measure of refresh success. The model also does not establish that refreshing a page will cause CTR to increase.

The error analysis shows that prediction difficulty varies across observations. The mean absolute error was **0.0202**, while the RMSE was **0.0527**. The maximum absolute error was **0.9971**, showing that a small number of observations can have very large prediction errors. Low-impression observations may produce unstable CTR values because a small change in clicks can cause a large change in the observed rate.

Permutation importance identified **gsc_avg_position, gsc_clicks, ga4_engaged_sessions, ga4_sessions, and ga4_total_engagement_sec** as the five most important observed features. These are associations used to interpret the model and should not be treated as causal effects.

Overall, the Random Forest is useful as a **directional decision-support ranking model**. It performs better than the Week-4 baseline on this held-out test period, but the result should not be interpreted as evidence that the model or a content refresh itself causes future performance improvements.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.